# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/Leem1nwon/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import json
from collections import Counter

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MixupTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import (
    collect_predictions, confusion_matrices, average_macro_f1,
    per_attribute_macro_f1, CLASS_NAMES,
)
from src.utils.submission import write_submission
from src.datasets.bdd_attr import (
    BDDAttrDataset, ATTRIBUTES, NUM_CLASSES, WEATHER_CLASSES,
)
# 우리 best 백본 = 직접 구현한 ViT-S/16 (라이브러리 모델 import 금지). resnet18 placeholder 교체.
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Colab `Run All` 재현 시 wandb 는 비활성화합니다 (조교 채점 환경 = WANDB_DISABLED=true).
# 본인이 직접 재학습하며 로깅하려면 이 줄을 주석 처리하고 아래 셀에서 RUN_TRAINING=True 로 두세요.
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# ===== 재현 모드 스위치 =====
# RUN_TRAINING = False → 재학습을 건너뛰고 동결 체크포인트(.pth)를 로드해 scoring/eval/제출/그림만 재현 (조교 채점 경로).
# RUN_TRAINING = True  → Set A + picks 로 처음부터 재학습 (H100 권장; T4 에서는 매우 오래 걸립니다).
RUN_TRAINING = False

WANDB_PROJECT = "aue8088-pa2" if RUN_TRAINING else None   # 재현 모드에서는 wandb 끔
STRATEGY_NAME = "hard-rarity-balanced"   # 본인 전략명 (Run 이름에 들어감)
WANDB_TAGS    = ["level5"]

CKPT_DIR = "../checkpoints"

In [ ]:
# --- 데이터셋 + 체크포인트 자동 다운로드 (Google Drive) -------------------------
# ../data/set_a, ../data/set_b 가 없으면 데이터 zip 을 받아 상위 폴더에 압축 해제.
# 재현 모드(RUN_TRAINING=False)면 동결 체크포인트 zip 도 받아 ../checkpoints 에 푼다.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"   # 데이터(set_a + set_b) zip
GDRIVE_CKPT_ID = "1B6XaYqchgb-wex0AHKyDJb3IF7eNcH54"          # ← 체크포인트 zip 의 Google Drive 파일 ID
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT  = "../data/set_a"
SET_B_ROOT = "../data/set_b"

def _ensure_gdown():
    try:
        import gdown  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    return gdown

# 1) 데이터 (Set A + Set B) — Level 5 는 Set B 풀이 반드시 필요
if not (os.path.isdir(DATA_ROOT) and os.path.isdir(SET_B_ROOT)):
    gdown = _ensure_gdown()
    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)
    print("데이터 압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}, {SET_B_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}, {SET_B_ROOT}")

# 2) 체크포인트 — 재현 모드에서 base(level3_best) + 재학습(level5_picks) 결과를 로드
#    base: {CKPT_DIR}/level3_best.pth  /  retrained: {CKPT_DIR}/level5_picks.pth
need_ckpt = not RUN_TRAINING and not (
    os.path.exists(f"{CKPT_DIR}/level3_best.pth") and os.path.exists(f"{CKPT_DIR}/level5_picks.pth")
)
if need_ckpt:
    if GDRIVE_CKPT_ID == "PUT_CHECKPOINT_ZIP_ID_HERE":
        print("[주의] GDRIVE_CKPT_ID 를 체크포인트 zip 파일 ID 로 교체하세요 "
              "(level3_best.pth, level5_picks.pth, tables/level5_*_metrics.json 포함).")
    else:
        gdown = _ensure_gdown()
        gdown.download(id=GDRIVE_CKPT_ID, output="../checkpoints.zip", quiet=False)
        with zipfile.ZipFile("../checkpoints.zip") as zf:
            zf.extractall("..")   # zip 내부 최상위가 checkpoints/, tables/ 이므로 상위에 푼다
        print("체크포인트/메트릭 다운로드·해제 완료")
else:
    print(f"체크포인트 준비 상태: RUN_TRAINING={RUN_TRAINING}")
# --------------------------------------------------------------------------

In [ ]:
# 1단계 — best base 모델(ViT, level3_best)로 Set B 의 모든 이미지를 score.
# base 백본은 우리 best 모델인 직접 구현한 ViT-S/16 (placeholder resnet18 교체).
base = vit_small_patch16_224().to(device)
base.load_state_dict(torch.load(f"{CKPT_DIR}/level3_best.pth", map_location="cpu")["state_dict"])
base.float().to(device).eval()

set_b = BDDAttrDataset(SET_B_ROOT, split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

preds_b, probs_b, _, ids_b = collect_predictions(base, loader_b, device)

# 이미지별 불확실성 (uncertainty) 계산: 1 - max-softmax 를 3개 head 평균. (노트북 정의)
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)
print(f"scored {len(ids_b)} | uncertainty shape={uncertainty.shape}, mean={uncertainty.mean():.3f}")

In [ ]:
# 2단계 — 선별 알고리즘 (scripts/level5_pick.py 의 balanced() 와 동일).
#
# 알고리즘 순서:
#   uncertainty(1단계) → rarity(Set A train 빈도역수, 속성별 [0,1] 정규화 후 3속성 평균)
#   → score = 0.5*uncertainty + 0.5*rarity → foggy 제외
#   → weather·scene·timeofday 3속성 모두 클래스 캡 하에 score 순 greedy 선택 (소수 클래스 독식 방지)
#
# Set B 의 정답 라벨은 set_b (split="mining") 에 채워져 있어 pseudo-label 로 사용한다.
LAM = 0.5
K = 1000
FOGGY = WEATHER_CLASSES.index("foggy")

STRATEGY = (
    "Hard-example + rarity + 3속성 균형. score = 0.5·uncertainty + 0.5·rarity. "
    "uncertainty = 1 − mean(max-softmax) (base=ViT best가 헷갈리는 샘플), "
    "rarity = Set A train 클래스 빈도 역수(속성별 [0,1] 정규화 후 3속성 평균; 소수 클래스/희귀 조합 보강). "
    "단 score top-K를 그대로 뽑으면 소수 클래스(dawn/dusk·residential 등 rarity·uncertainty 이중 최상위)가 "
    "독식하므로, weather·scene·timeofday 3속성 모두에 클래스 캡을 두고 score 순 greedy 선택하여 "
    "어느 한 클래스도 과반을 넘지 못하게 한다(multi-task 균형 설계). foggy는 Set B에도 0장이라 제외."
)

# rarity — Set A train inverse-frequency, 속성별로 [0,1] 정규화 (rarest class -> 1.0)
setA_train = BDDAttrDataset(DATA_ROOT, "train")
rar = {}
for a in ATTRIBUTES:
    c = setA_train.class_counts(a).float()
    inv = 1.0 / c.clamp(min=1)
    rar[a] = (inv / inv.max()).numpy()
rarity = np.array([
    np.mean([rar[a][getattr(s, a)] for a in ATTRIBUTES if getattr(s, a) >= 0])
    for s in set_b.samples
])

score = LAM * uncertainty + (1 - LAM) * rarity
foggy_mask = np.array([s.weather == FOGGY for s in set_b.samples])
score[foggy_mask] = -1.0   # exclude foggy (none anyway)

sw = np.array([s.weather   for s in set_b.samples])
ss = np.array([s.scene     for s in set_b.samples])
tod = np.array([s.timeofday for s in set_b.samples])
order = np.argsort(-score)
reason_fmt = f"score={{:.3f}} (unc·{LAM}+rar·{1-LAM}, 3-attr balanced)"

def balanced(n):
    # weather·scene·timeofday 3속성 모두 클래스 캡 → score 순 greedy (한 클래스 독식 방지)
    cap_w = int(n / (NUM_CLASSES["weather"] - 1) * 1.8)   # foggy 제외 5클래스
    cap_s = int(n / NUM_CLASSES["scene"] * 1.25)
    cap_t = int(n / NUM_CLASSES["timeofday"] * 1.25)
    cw, cs, ct = {}, {}, {}
    chosen = []
    for i in order:
        if foggy_mask[i]:
            continue
        w, s, t = int(sw[i]), int(ss[i]), int(tod[i])
        if cw.get(w, 0) >= cap_w or cs.get(s, 0) >= cap_s or ct.get(t, 0) >= cap_t:
            continue
        chosen.append(i); cw[w] = cw.get(w, 0) + 1; cs[s] = cs.get(s, 0) + 1; ct[t] = ct.get(t, 0) + 1
        if len(chosen) >= n:
            break
    if len(chosen) < n:   # 캡으로 미달 시 score 순 충원
        sel = set(chosen)
        for i in order:
            if i not in sel and not foggy_mask[i]:
                chosen.append(i)
                if len(chosen) >= n:
                    break
    return chosen

def picks_of(idxs):
    out = []
    for i in idxs:
        s = set_b.samples[i]
        out.append({
            "image_id": s.image_id,
            "weather":   int(s.weather),    # Set B 의 정답 라벨 (pseudo-label)
            "scene":     int(s.scene),
            "timeofday": int(s.timeofday),
            "reason":    reason_fmt.format(score[i]),
        })
    return out

picks = picks_of(balanced(K))
with open("../level5_picks.json", "w") as f:
    json.dump({"strategy": STRATEGY, "num_picks": len(picks), "picks": picks}, f, indent=2, ensure_ascii=False)
print(f"saved {len(picks)} picks → ../level5_picks.json")
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    print(f"  {a:10s}", {CLASS_NAMES[a][k]: cnt.get(k, 0) for k in range(NUM_CLASSES[a])})

In [ ]:
# 3단계 — Set A + picks 로 best 모델(ViT) 재학습 또는 동결 체크포인트 로드.
#   RUN_TRAINING=True  → ImageNet-pretrained init(load_pretrained_vit) + Mixup/CutMix,
#                        AdamW lr 1e-4 / wd 5e-2, 25 epochs, seed 42 (scripts/level5_retrain.py 와 동일).
#   RUN_TRAINING=False → 동결 ../checkpoints/level5_picks.pth 로드 (조교 채점 경로).
val_ds     = BDDAttrDataset(DATA_ROOT, "val", transform=eval_transform())
loader_val = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

if RUN_TRAINING:
    from scripts.vit_load_pretrained import load_pretrained_vit

    extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in picks]
    train_aug = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform(), extra_picks=extra)
    g = torch.Generator(); g.manual_seed(SEED)
    loader_tr = DataLoader(train_aug, batch_size=64, shuffle=True, num_workers=2,
                           worker_init_fn=seed_worker, generator=g, pin_memory=True)

    set_seed(SEED, deterministic=True)
    model = vit_small_patch16_224().to(device)
    load_pretrained_vit(model, verbose=False)   # ImageNet-pretrained init (Level 2 remap)
    optim  = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-2)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=25)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg    = TrainConfig(epochs=25, lr=1e-4, weight_decay=5e-2, amp=True)

    logger = WandbLogger(
        project=WANDB_PROJECT, run_name=f"level5-{STRATEGY_NAME}",
        config={"backbone": "vit_small_patch16_224", "pretrained": True, "aug": "mixup-cutmix",
                "strategy": STRATEGY_NAME, "n_extra": len(extra), "train_n": len(train_aug),
                "epochs": 25, "batch": 64, "lr": 1e-4, "weight_decay": 5e-2, "seed": SEED},
        tags=WANDB_TAGS + [STRATEGY_NAME])
    trainer = MixupTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(loader_tr, loader_val)

    val_pred, _, val_tgt, _ = collect_predictions(model, loader_val, device)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs(CKPT_DIR, exist_ok=True)
    torch.save({"state_dict": model.float().state_dict(), "history": history,
                "seed": SEED, "picks": "picks", "n_extra": len(extra)},
               f"{CKPT_DIR}/level5_picks.pth")
    print(f"재학습 완료 → {CKPT_DIR}/level5_picks.pth (n_extra={len(extra)})")
else:
    model = vit_small_patch16_224().to(device)
    model.load_state_dict(torch.load(f"{CKPT_DIR}/level5_picks.pth", map_location="cpu")["state_dict"])
    model.to(device).eval()
    print(f"체크포인트 로드 → {CKPT_DIR}/level5_picks.pth")

# val 성능 확인
val_pred, _, val_tgt, _ = collect_predictions(model, loader_val, device)
print(f"val Avg-MF1 = {average_macro_f1(val_pred, val_tgt):.4f}  per={per_attribute_macro_f1(val_pred, val_tgt)}")

In [ ]:
# 4단계 — Kaggle 제출용 CSV 생성 (Set A test, 라벨 비공개 → Kaggle LB 채점).
test_ds   = BDDAttrDataset(DATA_ROOT, "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

preds_te, _, _, ids_te = collect_predictions(model, loader_te, device)
os.makedirs("../submission", exist_ok=True)
write_submission("../submission/level5_submission.csv", ids_te, preds_te)
print("submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.")

In [ ]:
# 5단계 — Curation 분석: picks 분포 + DI 표 + 250/500/1000 ablation.
#
# DI 비교 메트릭은 동일 recipe 로 재학습한 결과를 tables/level5_*_metrics.json 에서 로드한다
# (재현 노트북에서 5개 run 을 재학습하지 않고 동결 메트릭을 읽음 — scripts/level5_retrain.py 산출).
#   DI = (Avg-MF1[picks] − Avg-MF1[random]) / Avg-MF1[random]
import matplotlib.pyplot as plt

TBL = "../tables"

# (1) picks 분포 시각화 — weather / scene / timeofday
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax_, a in zip(axes, ATTRIBUTES):
    cnt = Counter(p[a] for p in picks)
    names = CLASS_NAMES[a]
    vals = [cnt.get(k, 0) for k in range(NUM_CLASSES[a])]
    ax_.bar(range(len(names)), vals, color="steelblue")
    ax_.set_xticks(range(len(names))); ax_.set_xticklabels(names, rotation=30, ha="right")
    ax_.set_title(f"picks — {a}"); ax_.set_ylabel("count")
plt.suptitle("Level 5 picks 분포 (1,000장)"); plt.tight_layout(); plt.show()

# (2) DI 표 — setA_only(reference) / random(DI 분모) / picks(DI 분자), 250·500 ablation
def load_metrics(name):
    return json.loads(open(f"{TBL}/level5_{name}_metrics.json").read())

rows = {n: load_metrics(n) for n in ["setA_only", "random", "picks", "picks_250", "picks_500"]}
rand_mf1 = rows["random"]["final_val_avg_mf1"]

print(f"{'run':12s} {'n_extra':>8s} {'Avg-MF1':>9s} {'weather':>8s} {'scene':>7s} {'timeofday':>10s} {'DI vs random':>13s}")
for n in ["setA_only", "random", "picks", "picks_250", "picks_500"]:
    m = rows[n]; per = m["final_per_mf1"]; mf1 = m["final_val_avg_mf1"]
    di = (mf1 - rand_mf1) / rand_mf1
    di_s = f"{di:+.2%}" if n != "random" else "(분모)"
    print(f"{n:12s} {m['n_extra']:>8d} {mf1:>9.4f} {per['weather']:>8.3f} {per['scene']:>7.3f} {per['timeofday']:>10.3f} {di_s:>13s}")

di_final = (rows["picks"]["final_val_avg_mf1"] - rand_mf1) / rand_mf1
print(f"\nfinal DI (picks vs random-1000) = {di_final:+.2%}  "
      f"[picks {rows['picks']['final_val_avg_mf1']:.4f} vs random {rand_mf1:.4f}]")
print(f"한계 효용(ablation): 250→{rows['picks_250']['final_val_avg_mf1']:.4f}  "
      f"500→{rows['picks_500']['final_val_avg_mf1']:.4f}  "
      f"1000→{rows['picks']['final_val_avg_mf1']:.4f}")

## Curation Report — 요약

**선별 알고리즘** (의사코드 = 2단계 셀 `balanced()`):
`base(ViT) Set B 전수 scoring → uncertainty=1−mean(max-softmax) → rarity(Set A train 빈도역수, 속성별 정규화 평균) → score=0.5·unc+0.5·rar → foggy 제외 → weather·scene·timeofday 3속성 클래스 캡 하 score 순 greedy → 1,000장`.

**확정 결과** (백본 = 직접 구현 ViT-S/16, ImageNet-pretrained init + Mixup/CutMix, AdamW lr 1e-4 wd 5e-2 25ep, seed 42; 모든 run 동일 recipe → 차이는 picks 만):

| run | n_extra | val Avg-MF1 | DI vs random |
|---|---|---|---|
| setA_only (reference) | 0 | 0.7007 | −1.06% |
| random-1000 (DI 분모) | 1000 | 0.7082 | — |
| **picks-1000 (제출)** | 1000 | **0.7240** | **+2.22%** |
| picks-250 (ablation) | 250 | 0.7196 | +1.61% |
| picks-500 (ablation) | 500 | 0.7128 | +0.65% |

- **DI = +2.22%** (picks 0.7240 vs random 0.7082). picks 는 setA_only(0.7007) 대비도 +3.32%.
- **Ablation**: 동일 큐레이션이라도 250→500→1000 으로 갈수록 한계 효용이 단조롭지 않음(250 이 500 보다 높게 나온 구간 존재) — Curation Report 에 추가 데이터의 한계 효용 곡선으로 논의.
- 위 표·DI 는 동결 `tables/level5_*_metrics.json` 에서 로드(재현 노트북은 재학습 없이 inference·메트릭만). 직접 재학습하려면 `RUN_TRAINING=True`.

**산출물**: `level5_picks.json` (선택 image_id + 메타데이터/reason) · `submission/level5_submission.csv`.